<a href="https://colab.research.google.com/github/SasmitDey/financial_fraud_detector/blob/main/fin_fraud_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com
!pip -q install polars[gpu] cupy-cuda12x plotly kagglehub

In [16]:
import polars as pl
import kagglehub
import cupy as cp

#DATASET

_We use the paysim dataset for this clustering task_

In [3]:
path = kagglehub.dataset_download("ealaxi/paysim1")

Using Colab cache for faster access to the 'paysim1' dataset.


In [4]:
df = pl.read_csv("/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv")

In [5]:
df.describe()

statistic,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
str,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64
"""count""",6.36262e6,"""6362620""",6.36262e6,"""6362620""",6.36262e6,6.36262e6,"""6362620""",6.36262e6,6.36262e6,6.36262e6,6.36262e6
"""null_count""",0.0,"""0""",0.0,"""0""",0.0,0.0,"""0""",0.0,0.0,0.0,0.0
"""mean""",243.397246,null,179861.903549,null,833883.104074,855113.668579,null,1.1007e6,1.2250e6,0.001291,0.000003
"""std""",142.331971,null,603858.231463,null,2.8882e6,2.9240e6,null,3.3992e6,3.6741e6,0.035905,0.001586
"""min""",1.0,"""CASH_IN""",0.0,"""C1000000639""",0.0,0.0,"""C1000004082""",0.0,0.0,0.0,0.0
"""25%""",156.0,null,13389.57,null,0.0,0.0,null,0.0,0.0,0.0,0.0
"""50%""",239.0,null,74872.08,null,14208.0,0.0,null,132705.81,214661.65,0.0,0.0
"""75%""",335.0,null,208721.45,null,107315.0,144258.41,null,943036.53,1.1119e6,0.0,0.0
"""max""",743.0,"""TRANSFER""",9.2446e7,"""C999999784""",5.9585e7,4.9585e7,"""M999999784""",3.5602e8,3.5618e8,1.0,1.0


In [6]:
df.head()

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
i64,str,f64,str,f64,f64,str,f64,f64,i64,i64
1,"""PAYMENT""",9839.64,"""C1231006815""",170136.0,160296.36,"""M1979787155""",0.0,0.0,0,0
1,"""PAYMENT""",1864.28,"""C1666544295""",21249.0,19384.72,"""M2044282225""",0.0,0.0,0,0
1,"""TRANSFER""",181.0,"""C1305486145""",181.0,0.0,"""C553264065""",0.0,0.0,1,0
1,"""CASH_OUT""",181.0,"""C840083671""",181.0,0.0,"""C38997010""",21182.0,0.0,1,0
1,"""PAYMENT""",11668.14,"""C2048537720""",41554.0,29885.86,"""M1230701703""",0.0,0.0,0,0


#FEATURE ENGINEERING

_We make new columns based on matching the expected amount with received amount for the destination_

In [7]:
df = df.with_columns([
    (pl.col("oldbalanceOrg") - pl.col("amount") != pl.col("newbalanceOrig")).alias("mismatch_orig"),
    (pl.col("newbalanceDest") - pl.col("oldbalanceDest") != pl.col("amount")).alias("mismatch_dest"),
])
df = df.to_dummies(columns="type")

df = df.select(pl.all().exclude(["nameOrig", "nameDest", "type"])).cast(pl.Float32)

In [8]:
df.head()

step,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,mismatch_orig,mismatch_dest
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
1.0,0.0,0.0,0.0,1.0,0.0,9839.639648,170136.0,160296.359375,0.0,0.0,0.0,0.0,0.0,1.0
1.0,0.0,0.0,0.0,1.0,0.0,1864.280029,21249.0,19384.720703,0.0,0.0,0.0,0.0,0.0,1.0
1.0,0.0,0.0,0.0,0.0,1.0,181.0,181.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1.0,0.0,1.0,0.0,0.0,0.0,181.0,181.0,0.0,21182.0,0.0,1.0,0.0,0.0,1.0
1.0,0.0,0.0,0.0,1.0,0.0,11668.139648,41554.0,29885.859375,0.0,0.0,0.0,0.0,0.0,1.0


In [9]:
X = df.to_numpy()

#MODEL FITTING

_IsolationForest is used to find anomalies_

In [10]:
%load_ext cuml.accel
from sklearn.ensemble import IsolationForest

model=IsolationForest(contamination=0.001)
model.fit(X)

from cuml.accel import is_proxy
print(f"Is the model running on GPU? {is_proxy(model)}")

Is the model running on GPU? False


In [14]:
df = df.with_columns([
    pl.Series("is_anomaly", model.predict(X)),
    pl.Series("anomaly_score", model.decision_function(X))
])

#INFERENCE

__We check how many anomalies we found.__

__We've found 6363 anomalies from the entire dataset of ~6 million rows.__

__We then show the top 10 anomalous cases sorted by their anomaly score, i.e., these are the top 10 points that stick out the most__

In [20]:
print(df["is_anomaly"].value_counts())


top_fraud = df.sort("anomaly_score").head(10)
print(top_fraud)

shape: (2, 2)
┌────────────┬─────────┐
│ is_anomaly ┆ count   │
│ ---        ┆ ---     │
│ i64        ┆ u32     │
╞════════════╪═════════╡
│ -1         ┆ 6363    │
│ 1          ┆ 6356257 │
└────────────┴─────────┘
shape: (10, 17)
┌───────┬────────────┬────────────┬────────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ step  ┆ type_CASH_ ┆ type_CASH_ ┆ type_DEBIT ┆ … ┆ mismatch_ ┆ mismatch_ ┆ is_anomal ┆ anomaly_s │
│ ---   ┆ IN         ┆ OUT        ┆ ---        ┆   ┆ orig      ┆ dest      ┆ y         ┆ core      │
│ f32   ┆ ---        ┆ ---        ┆ f32        ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│       ┆ f32        ┆ f32        ┆            ┆   ┆ f32       ┆ f32       ┆ i64       ┆ f64       │
╞═══════╪════════════╪════════════╪════════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 730.0 ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ … ┆ 0.0       ┆ 1.0       ┆ -1        ┆ -0.066101 │
│ 730.0 ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ … ┆ 0.0       

#UMAP

__We use UMAP from cuML library by RAPIDS. We filter out the anomalies, and represent them in the plot__

In [21]:
from cuml.manifold import UMAP
import cupy as cp

anomalies_only = df.filter(pl.col("is_anomaly") == -1)

X_anom_gpu = cp.asarray(
    anomalies_only.drop(["is_anomaly", "anomaly_score"]).to_numpy(),
    dtype=cp.float32
)

reducer = UMAP(n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(X_anom_gpu)

viz_df = anomalies_only.with_columns([
    pl.Series("x", embedding[:, 0]),
    pl.Series("y", embedding[:, 1])
])

#VISUALISATION

__We used plotly to represent these anomalies, along with the patterns that further helps us infer what kind of activities could have taken place in these anomalous transactions__

In [22]:
import plotly.express as px

fig = px.scatter(
    viz_df.to_pandas(),
    x="x",
    y="y",
    color="amount",          # Shows the financial "weight" of the transactions
    size="amount",           # Larger dots = larger sums of money
    hover_data=["mismatch_orig", "mismatch_dest", "step"], # Audit details
    title="Dashboard: Uncovering Financial Fraud Rings",
    color_continuous_scale=px.colors.sequential.Viridis,
    template="plotly_dark"   # Dark mode makes the clusters pop
)

fig.update_layout(
    xaxis_title="UMAP Dimension 1",
    yaxis_title="UMAP Dimension 2",
    coloraxis_colorbar=dict(title="Transaction Amount")
)

fig.show()

In [23]:
whale_ring = viz_df.filter((pl.col("x") < -7) & (pl.col("y") < -15))
print(whale_ring.describe())

shape: (9, 20)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ step      ┆ type_CASH ┆ type_CASH ┆ … ┆ is_anomal ┆ anomaly_s ┆ x         ┆ y        │
│ ---       ┆ ---       ┆ _IN       ┆ _OUT      ┆   ┆ y         ┆ core      ┆ ---       ┆ ---      │
│ str       ┆ f64       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ f64       ┆ f64      │
│           ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 120.0     ┆ 120.0     ┆ 120.0     ┆ … ┆ 120.0     ┆ 120.0     ┆ 120.0     ┆ 120.0    │
│ null_coun ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ t         ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ mean      ┆ 316.55831 ┆ 0.083333  ┆ 0.008333  ┆ … ┆ -1.0      ┆ -0.018207 

In [24]:
print(whale_ring["amount"].sum())

3298183168.0


__3.2 BILLION was the total amount of money moved by these laundering rings__